# Build a regression model using Scikit-learn: prepare and visualize data

**ML2 · Regression & Deployment** — [AI Olympiad Curriculum](https://abzlitochka.github.io/ai-olympiad-curriculum)

~19 min

## Introduction

![Data visualization infographic](ms-assets/redrawn/data-visualization.svg)

Infographic by [Dasani Madipalli](https://twitter.com/dasani_decoded)


> ### [This lesson is available in R!](https://github.com/microsoft/ML-For-Beginners/blob/main/2-Regression/2-Data/solution/R/lesson_2.html)

Now that you are set up with the tools you need to start tackling machine learning model building with Scikit-learn, you are ready to start asking questions of your data. As you work with data and apply ML solutions, it's very important to understand how to ask the right question to properly unlock the potentials of your dataset.

In this lesson, you will learn:

- How to prepare your data for model-building.
- How to use Matplotlib for data visualization.
- How to use Seaborn for more expressive data visualization.



## Asking the right question of your data

The question you need answered will determine what type of ML algorithms you will leverage. And the quality of the answer you get back will be heavily dependent on the nature of your data.

Take a look at the [data](https://github.com/microsoft/ML-For-Beginners/blob/main/2-Regression/data/US-pumpkins.csv) provided for this lesson. You can open this .csv file in VS Code. A quick skim immediately shows that there are blanks and a mix of strings and numeric data. There's also a strange column called 'Package' where the data is a mix between 'sacks', 'bins' and other values. The data, in fact, is a bit of a mess.

> [Video](https://youtu.be/5qGjczWTrDQ)

In fact, it is not very common to be gifted a dataset that is completely ready to use to create a ML model out of the box. In this lesson, you will learn how to prepare a raw dataset using standard Python libraries. You will also learn various techniques to visualize the data.



## Case study: 'the pumpkin market'

In this folder you will find a .csv file in the root `data` folder called [US-pumpkins.csv](https://github.com/microsoft/ML-For-Beginners/blob/main/2-Regression/data/US-pumpkins.csv) which includes 1757 lines of data about the market for pumpkins, sorted into groupings by city. This is raw data extracted from the [Specialty Crops Terminal Markets Standard Reports](https://www.marketnews.usda.gov/mnp/fv-report-config-step1?type=termPrice) distributed by the United States Department of Agriculture.



## Preparing data

This data is in the public domain. It can be downloaded in many separate files, per city, from the USDA web site. To avoid too many separate files, we have concatenated all the city data into one spreadsheet, thus we have already _prepared_ the data a bit. Next, let's take a closer look at the data.



## The pumpkin data - early conclusions

What do you notice about this data? You already saw that there is a mix of strings, numbers, blanks and strange values that you need to make sense of.

What question can you ask of this data, using a Regression technique? What about "Predict the price of a pumpkin for sale during a given month". Looking again at the data, there are some changes you need to make to create the data structure necessary for the task.



## Exercise - analyze the pumpkin data

Let's use [Pandas](https://pandas.pydata.org/), (the name stands for `Python Data Analysis`) a tool very useful for shaping data, to analyze and prepare this pumpkin data.



## First, check for missing dates

You will first need to take steps to check for missing dates:

1. Convert the dates to a month format (these are US dates, so the format is `MM/DD/YYYY`).
2. Extract the month to a new column.

Open the _notebook.ipynb_ file in Visual Studio Code and import the spreadsheet in to a new Pandas dataframe.

1. Use the `head()` function to view the first five rows.

    ```python
    import pandas as pd
    pumpkins = pd.read_csv('https://raw.githubusercontent.com/microsoft/ML-For-Beginners/main/2-Regression/data/US-pumpkins.csv')
    pumpkins.head()
    ```

    ✅ What function would you use to view the last five rows?

1. Check if there is missing data in the current dataframe:

    ```python
    pumpkins.isnull().sum()
    ```

    There is missing data, but maybe it won't matter for the task at hand.

1. To make your dataframe easier to work with, select only the columns you need, using the `loc` function which extracts from the original dataframe a group of rows (passed as first parameter) and columns (passed as second parameter). The expression `:` in the case below means "all rows".

    ```python
    columns_to_select = ['Package', 'Low Price', 'High Price', 'Date']
    pumpkins = pumpkins.loc[:, columns_to_select]
    ```



## Second, determine average price of pumpkin

Think about how to determine the average price of a pumpkin in a given month. What columns would you pick for this task? Hint: you'll need 3 columns.

Solution: take the average of the `Low Price` and `High Price` columns to populate the new Price column, and convert the Date column to only show the month. Fortunately, according to the check above, there is no missing data for dates or prices.

1. To calculate the average, add the following code:

    ```python
    price = (pumpkins['Low Price'] + pumpkins['High Price']) / 2

    month = pd.DatetimeIndex(pumpkins['Date']).month

    ```

   ✅ Feel free to print any data you'd like to check using `print(month)`.

2. Now, copy your converted data into a fresh Pandas dataframe:

    ```python
    new_pumpkins = pd.DataFrame({'Month': month, 'Package': pumpkins['Package'], 'Low Price': pumpkins['Low Price'],'High Price': pumpkins['High Price'], 'Price': price})
    ```

    Printing out your dataframe will show you a clean, tidy dataset on which you can build your new regression model.



## But wait! There's something odd here

If you look at the `Package` column, pumpkins are sold in many different configurations. Some are sold in '1 1/9 bushel' measures, and some in '1/2 bushel' measures, some per pumpkin, some per pound, and some in big boxes with varying widths.

> Pumpkins seem very hard to weigh consistently

Digging into the original data, it's interesting that anything with `Unit of Sale` equalling 'EACH' or 'PER BIN' also have the `Package` type per inch, per bin, or 'each'. Pumpkins seem to be very hard to weigh consistently, so let's filter them by selecting only pumpkins with the string 'bushel' in their `Package` column.

1. Add a filter at the top of the file, under the initial .csv import:

    ```python
    pumpkins = pumpkins[pumpkins['Package'].str.contains('bushel', case=True, regex=True)]
    ```

    If you print the data now, you can see that you are only getting the 415 or so rows of data containing pumpkins by the bushel.



## But wait! There's one more thing to do

Did you notice that the bushel amount varies per row? You need to normalize the pricing so that you show the pricing per bushel, so do some math to standardize it.

1. Add these lines after the block creating the new_pumpkins dataframe:

    ```python
    new_pumpkins.loc[new_pumpkins['Package'].str.contains('1 1/9'), 'Price'] = price/(1 + 1/9)

    new_pumpkins.loc[new_pumpkins['Package'].str.contains('1/2'), 'Price'] = price/(1/2)
    ```

✅ According to [The Spruce Eats](https://www.thespruceeats.com/how-much-is-a-bushel-1389308), a bushel's weight depends on the type of produce, as it's a volume measurement. "A bushel of tomatoes, for example, is supposed to weigh 56 pounds... Leaves and greens take up more space with less weight, so a bushel of spinach is only 20 pounds." It's all pretty complicated! Let's not bother with making a bushel-to-pound conversion, and instead price by the bushel. All this study of bushels of pumpkins, however, goes to show how very important it is to understand the nature of your data!

Now, you can analyze the pricing per unit based on their bushel measurement. If you print out the data one more time, you can see how it's standardized.

✅ Did you notice that pumpkins sold by the half-bushel are very expensive? Can you figure out why? Hint: little pumpkins are way pricier than big ones, probably because there are so many more of them per bushel, given the unused space taken by one big hollow pie pumpkin.



## Visualization Strategies

Part of the data scientist's role is to demonstrate the quality and nature of the data they are working with. To do this, they often create interesting visualizations, or plots, graphs, and charts, showing different aspects of data. In this way, they are able to visually show relationships and gaps that are otherwise hard to uncover.

> [Video](https://youtu.be/SbUkxH6IJo0)

Visualizations can also help determine the machine learning technique most appropriate for the data. A scatterplot that seems to follow a line, for example, indicates that the data is a good candidate for a linear regression exercise.

One data visualization library that works well in Jupyter notebooks is [Matplotlib](https://matplotlib.org/) (which you also saw in the previous lesson).

> Get more experience with data visualization in [these tutorials](https://learn.microsoft.com/learn/modules/explore-analyze-data-with-python?WT.mc_id=academic-77952-leestott).



## Exercise - experiment with Matplotlib

Try to create some basic plots to display the new dataframe you just created. What would a basic line plot show?

1. Import Matplotlib at the top of the file, under the Pandas import:

    ```python
    import matplotlib.pyplot as plt
    ```

1. Rerun the entire notebook to refresh.
1. At the bottom of the notebook, add a cell to plot the data as a box:

    ```python
    price = new_pumpkins.Price
    month = new_pumpkins.Month
    plt.scatter(price, month)
    plt.show()
    ```

    ![A scatterplot showing price to month relationship](https://raw.githubusercontent.com/microsoft/ML-For-Beginners/main/2-Regression/2-Data/images/scatterplot.png)

    Is this a useful plot? Does anything about it surprise you?

    It's not particularly useful as all it does is display in your data as a spread of points in a given month.



## Make it useful

To get charts to display useful data, you usually need to group the data somehow. Let's try creating a plot where the y axis shows the months and the data demonstrates the distribution of data.

1. Add a cell to create a grouped bar chart:

    ```python
    new_pumpkins.groupby(['Month'])['Price'].mean().plot(kind='bar')
    plt.ylabel("Pumpkin Price")
    ```

    ![A bar chart showing price to month relationship](https://raw.githubusercontent.com/microsoft/ML-For-Beginners/main/2-Regression/2-Data/images/barchart.png)

    This is a more useful data visualization! It seems to indicate that the highest price for pumpkins occurs in September and October. Does that meet your expectation? Why or why not?



## Exercise - experiment with Seaborn

Matplotlib is powerful, but it can take a lot of code to produce a polished chart. [Seaborn](https://seaborn.pydata.org/) is a library built _on top of_ Matplotlib that is designed for statistical data visualization. It works directly with Pandas dataframes, applies attractive default styles, and lets you create informative plots with far less code. Because Seaborn returns Matplotlib objects, you can still use everything you already know about Matplotlib to fine-tune the result.

> If you don't already have Seaborn installed, install it with `pip install seaborn`.

1. Import Seaborn at the top of the notebook, under the other imports. It is conventionally imported as `sns`:

    ```python
    import seaborn as sns
    ```



## Scatter plots to show relationships

A big part of exploring data before building a model is looking for _relationships_ between variables. A [scatter plot](https://en.wikipedia.org/wiki/Scatter_plot) is one of the best tools for this: if the points seem to follow a line, the two variables may be correlated, which is a good sign that a linear regression model could work.

1. Recreate the price-to-month scatter plot from before, this time using Seaborn's [`relplot()`](https://seaborn.pydata.org/generated/seaborn.relplot.html) (relational plot), which works directly with your dataframe columns:

    ```python
    sns.relplot(x="Price", y="Month", data=new_pumpkins)
    ```

    ![A Seaborn scatterplot showing price to month relationship](https://raw.githubusercontent.com/microsoft/ML-For-Beginners/main/2-Regression/2-Data/images/relplot.png)

    Notice how you pass the _column names_ and the dataframe, and Seaborn takes care of the axis labels for you.

2. You can switch to a line plot by passing `kind="line"`. Seaborn even draws a shaded band showing the confidence interval around the line:

    ```python
    sns.relplot(x="Price", y="Month", kind="line", data=new_pumpkins)
    ```

    ![A Seaborn line plot showing price to month relationship](https://raw.githubusercontent.com/microsoft/ML-For-Beginners/main/2-Regression/2-Data/images/lineplot.png)

    This particular data is quite noisy, so a line plot isn't the clearest choice here — but it shows how easily you can change chart types in Seaborn.



## Bar charts to show distributions

Earlier you grouped the data by hand to create a bar chart with Matplotlib. Seaborn's [`catplot()`](https://seaborn.pydata.org/generated/seaborn.catplot.html) (categorical plot) can do the grouping and aggregation for you. By default `kind="bar"` shows the mean of each category along with a black line indicating the confidence interval.

1. Create a bar chart of average price per month:

    ```python
    sns.catplot(x="Month", y="Price", data=new_pumpkins, kind="bar")
    ```

    ![A Seaborn bar chart showing the price distribution per month](https://raw.githubusercontent.com/microsoft/ML-For-Beginners/main/2-Regression/2-Data/images/catplot.png)

    This confirms what you saw with Matplotlib — prices peak around September and October — but Seaborn also visualizes how much the price _varies_ within each month.



## Heatmaps to show correlations

Scatter plots compare two variables at a time. When you have several numeric columns, a [heatmap](https://en.wikipedia.org/wiki/Heat_map) lets you view the strength of the relationship between _every_ pair of columns at once. This is a common way to spot which features are most correlated before choosing what to feed into a model (and the same kind of chart is later used to display confusion matrices in classification).

1. Build a correlation matrix with Pandas, then draw it with Seaborn's [`heatmap()`](https://seaborn.pydata.org/generated/seaborn.heatmap.html). The `annot=True` option prints the correlation values on each cell:

    ```python
    correlations = new_pumpkins[['Month', 'Low Price', 'High Price', 'Price']].corr()
    sns.heatmap(correlations, annot=True, cmap="coolwarm")
    ```

    ![A Seaborn heatmap showing correlations between the numeric columns](https://raw.githubusercontent.com/microsoft/ML-For-Beginners/main/2-Regression/2-Data/images/heatmap.png)

    Values close to `1` (or `-1`) mean the columns are strongly _linearly_ correlated. Notice how `Low Price` and `High Price` are almost perfectly correlated. `Month`, on the other hand, shows only a weak linear correlation with price — even though the bar chart above revealed a clear seasonal peak in September and October. That's an important lesson: the correlation coefficient only measures _straight-line_ relationships, so it can miss seasonal or otherwise non-linear patterns. ✅ Why is it useful to look at both a heatmap *and* charts like the bar chart before deciding which columns to use?



## Matplotlib or Seaborn?

Both libraries are worth knowing:

- **Matplotlib** gives you fine-grained control over every element of a chart and is the foundation almost every other Python plotting library builds on.
- **Seaborn** provides higher-level functions and attractive defaults for statistical charts, works directly with dataframes, and is often quicker for exploratory data analysis.

A common workflow is to reach for Seaborn to explore your data quickly, then drop down to Matplotlib when you need to customize the details.



## Challenge

Explore the different types of visualization that Matplotlib and Seaborn offer. Which types are most appropriate for regression problems?



## Review & Self Study

Take a look at the many ways to visualize data. Make a list of the various libraries available and note which are best for given types of tasks, for example 2D visualizations vs. 3D visualizations. What do you discover?



## Assignment

[Exploring visualization](https://github.com/microsoft/ML-For-Beginners/blob/main/2-Regression/2-Data/assignment.md)



---

## ✅ Self-check tasks

Implement the function in the 'your code' cell, then run the check cell — it runs randomized tests and shows the first mismatch. Run the helper cell below once first.

1. [Median imputation of NaNs](https://abzlitochka.github.io/ai-olympiad-curriculum/#ex=m01-median-impute)
2. [One-hot encode labels](https://abzlitochka.github.io/ai-olympiad-curriculum/#ex=m00-onehot)
3. [Z-score using train statistics](https://abzlitochka.github.io/ai-olympiad-curriculum/#ex=m01-zscore-apply)

In [ ]:
# ── task-checking helper cell — run once ──
import copy
import numpy as np
import math, statistics, itertools, collections, functools, heapq, random

try:
    import sys; sys.stdout.reconfigure(encoding="utf-8", errors="replace")
except Exception:
    pass

_L = {"noFn":"❓ Define a function `{e}` in the solution cell and run it","runErr":"❌ Test {i}: {t}: {m}","wrong":"❌ Test {i}: {r}","badRef":"⚠️ Check harness broke (generator/reference): {t}: {m}","inL":"Input","expL":"Expected","gotL":"Got","ok":"✅ Correct! All {n} tests passed."}

def _nb_copy_kw(kw):
    return {k: (v.copy() if isinstance(v, np.ndarray) else copy.deepcopy(v)) for k, v in kw.items()}

def _nb_jsafe(x, depth=0):
    if depth > 6: return "…"
    if isinstance(x, np.ndarray): return _nb_jsafe(x.tolist(), depth + 1)
    if isinstance(x, np.integer): return int(x)
    if isinstance(x, np.floating): return float(x)
    if isinstance(x, np.bool_): return bool(x)
    if isinstance(x, float): return round(x, 6) if math.isfinite(x) else str(x)
    if isinstance(x, (list, tuple)): return [_nb_jsafe(v, depth + 1) for v in x]
    if isinstance(x, dict): return {str(k): _nb_jsafe(v, depth + 1) for k, v in x.items()}
    if isinstance(x, (int, str, bool)) or x is None: return x
    return repr(x)

def _nb_cmp(exp, got, path=""):
    if isinstance(exp, np.ndarray) or isinstance(got, np.ndarray):
        e, g = np.asarray(exp), np.asarray(got)
        if e.shape != g.shape:
            return f"shape {tuple(g.shape)} != expected {tuple(e.shape)}"
        try:
            if np.issubdtype(e.dtype, np.number) or np.issubdtype(g.dtype, np.number):
                ef, gf = e.astype(float), g.astype(float)
                close = np.isclose(gf, ef, atol=1e-6, rtol=1e-5, equal_nan=True)
                if not bool(close.all()):
                    i = tuple(int(v) for v in np.argwhere(~close)[0])
                    return f"{path}first mismatch at index {i}: got {gf[i]:.6g} vs expected {ef[i]:.6g}"
            elif not np.array_equal(g, e):
                return f"{path}values differ"
        except Exception as ex:
            return f"{path}cannot compare: {ex}"
        return None
    if isinstance(exp, (int, float)) and not isinstance(exp, bool) and isinstance(got, (int, float)) and not isinstance(got, bool):
        if not np.isclose(float(exp), float(got), atol=1e-6, rtol=1e-5):
            return f"{path}{got} != expected {exp}"
        return None
    if isinstance(exp, (list, tuple)) and isinstance(got, (list, tuple)):
        if len(exp) != len(got):
            return f"{path}length {len(got)} != expected {len(exp)}"
        for i, (a, b) in enumerate(zip(exp, got)):
            r = _nb_cmp(a, b, path + f"[{i}].")
            if r: return r
        return None
    if isinstance(exp, dict) and isinstance(got, dict):
        if set(exp) != set(got):
            return f"{path}keys differ"
        for k in exp:
            r = _nb_cmp(exp[k], got[k], path + f"[{k!r}].")
            if r: return r
        return None
    if exp != got:
        return f"{path}{got!r} != expected {exp!r}"
    return None

def _check_task(g, ref_src, entry, n=20, seed=0):
    """g — globals() ноутбука; ref_src — gen() + эталонное решение."""
    fn = g.get(entry)
    if not callable(fn):
        print(_L["noFn"].format(e=entry)); return
    ns = {"np": np, "math": math, "statistics": statistics, "itertools": itertools,
          "collections": collections, "functools": functools, "heapq": heapq, "random": random}
    try:
        exec(ref_src, ns)
        gen, ref = ns["gen"], ns[entry]
    except Exception as e:
        print(_L["badRef"].format(t=type(e).__name__, m=e)); return
    rng = np.random.default_rng(seed)
    for i in range(n):
        try:
            kw = gen(rng)
        except Exception as e:
            print(_L["badRef"].format(t=type(e).__name__, m=e)); return
        try:
            got = fn(**_nb_copy_kw(kw))
        except Exception as e:
            print(_L["runErr"].format(i=i + 1, t=type(e).__name__, m=e)
                  + "\n" + _L["inL"] + ": " + repr(_nb_jsafe(kw))); return
        exp = ref(**_nb_copy_kw(kw))
        r = _nb_cmp(exp, got)
        if r:
            print(_L["wrong"].format(i=i + 1, r=r)
                  + "\n" + _L["inL"] + ": " + repr(_nb_jsafe(kw))
                  + "\n" + _L["expL"] + ": " + repr(_nb_jsafe(exp))
                  + "\n" + _L["gotL"] + ": " + repr(_nb_jsafe(got))); return
    print(_L["ok"].format(n=n))

### Task 1 — Median imputation of NaNs

●○○ · ~10 min · [Open in the trainer →](https://abzlitochka.github.io/ai-olympiad-curriculum/#ex=m01-median-impute)

> Real olympiad tables are full of missing values, and `np.nan` poisons every reduction it touches (`np.mean` of a column with one NaN is NaN). Median imputation is the robust default: the median ignores outliers, unlike the mean. `np.nanmedian(X, axis=0)` computes per-column medians skipping NaNs; then you fill the missing cells column by column — e.g. via `np.where(np.isnan(X))` indices.

Implement `impute_median(X)` → a copy of `X` (float 2-D array) where every `NaN` is replaced by the median of its column. Do not modify the input in place. Every column is guaranteed to have at least one non-NaN value.

```
[[1, nan], [3, 2], [5, 4]] → [[1, 3], [3, 2], [5, 4]]   (col medians: 3, 3)
```

<details><summary>Hint</summary>

med = np.nanmedian(X, axis=0); rows, cols = np.where(np.isnan(X)); out[rows, cols] = med[cols].

</details>


In [ ]:
import numpy as np

def impute_median(X):
    X = np.asarray(X, dtype=float)
    out = X.copy()
    # your code
    return out

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    n, d = int(rng.integers(6, 20)), int(rng.integers(2, 6))\n    X = rng.normal(0, 3, (n, d))\n    mask = rng.random((n, d)) < 0.25\n    for j in range(d):\n        if mask[:, j].all():\n            mask[int(rng.integers(0, n)), j] = False   # keep >=1 observed per column\n    X[mask] = np.nan\n    return {\"X\": X}\n\ndef impute_median(X):\n    X = np.asarray(X, dtype=float)\n    out = X.copy()\n    med = np.nanmedian(out, axis=0)\n    rows, cols = np.where(np.isnan(out))\n    out[rows, cols] = med[cols]\n    return out", "impute_median", 20)

### Task 2 — One-hot encode labels

●○○ · ~8 min · [Open in the trainer →](https://abzlitochka.github.io/ai-olympiad-curriculum/#ex=m00-onehot)

> One-hot encoding turns a label vector into an (n, k) matrix of 0/1. One-line version: `np.eye(k)[y]` indexes the identity matrix by the labels. You will reuse this constantly.

Implement `one_hot(y, k)` → (n, k) float array, 1 at the label index.

<details><summary>Hint</summary>

np.eye(k)[y] does it in one line.

</details>


In [ ]:
import numpy as np

def one_hot(y, k):
    # your code
    return np.zeros((len(y), k))

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    n = int(rng.integers(3, 30)); k = int(rng.integers(2, 7))\n    hi = k - 1 if rng.random() < 0.35 else k   # sometimes class k-1 absent -> k must come from the arg\n    return {\"y\": rng.integers(0, hi, n), \"k\": k}\n\ndef one_hot(y, k):\n    return np.eye(k)[np.asarray(y)]", "one_hot", 20)

### Task 3 — Z-score using train statistics

●○○ · ~10 min · [Open in the trainer →](https://abzlitochka.github.io/ai-olympiad-curriculum/#ex=m01-zscore-apply)

> Standardization subtracts the column mean and divides by the column standard deviation. The subtle rule that separates good pipelines from leaky ones: compute `mu` and `sigma` on the TRAIN set only, then apply those same numbers to validation/test. Using `X.std(axis=0)` gives the population std (ddof=0). A constant train column has sigma=0 — the convention is to divide by 1 instead, which maps it to 0.

Implement `zscore_apply(X_train, X_test)`: compute per-column mean and population std (`ddof=0`) **on `X_train`**, and return `X_test` standardized with them. If a train column has std exactly 0, divide by 1 — the train column then maps to 0, and test values keep their raw offset from the train mean.

<details><summary>Hint</summary>

mu = X_train.mean(0); sd = np.where(X_train.std(0) == 0, 1.0, X_train.std(0)); then (X_test - mu) / sd broadcasts.

</details>


In [ ]:
import numpy as np

def zscore_apply(X_train, X_test):
    X_train = np.asarray(X_train, dtype=float)
    X_test = np.asarray(X_test, dtype=float)
    # your code — stats from X_train only!
    return X_test

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    n1, n2, d = int(rng.integers(8, 30)), int(rng.integers(3, 15)), int(rng.integers(2, 6))\n    X_train = rng.normal(0, 1, (n1, d)) * rng.integers(1, 5, d)\n    X_test = rng.normal(0, 1, (n2, d)) * rng.integers(1, 5, d)\n    if rng.random() < 0.4:\n        X_train[:, int(rng.integers(0, d))] = 2.5   # constant column\n    return {\"X_train\": X_train, \"X_test\": X_test}\n\ndef zscore_apply(X_train, X_test):\n    X_train = np.asarray(X_train, dtype=float)\n    X_test = np.asarray(X_test, dtype=float)\n    mu = X_train.mean(axis=0)\n    sd = X_train.std(axis=0)\n    sd = np.where(sd == 0, 1.0, sd)\n    return (X_test - mu) / sd", "zscore_apply", 20)

---

**Go deeper**
- [Original lesson (Microsoft, MIT)](https://github.com/microsoft/ML-For-Beginners/blob/main/2-Regression/2-Data)
- [Yandex ML handbook (RU): supervised learning](https://contest.yandex.ru/tracks/ml/supervised-learning/introduction)
- [Yandex ML handbook (RU): bias and variance](https://contest.yandex.ru/tracks/ml/machine-learning-theory/bias-variance-decomposition)
- [Yandex math handbook (RU): linear algebra](https://contest.yandex.ru/tracks/math/linear-algebra/chapter-overview)
- [Yandex math handbook (RU): gradients and optimization](https://contest.yandex.ru/tracks/math/mathematical-analysis/chapter-overview)